# Clasificador Naive Bayes para Géneros Literarios

El clasificador **Naive Bayes** es un algoritmo de clasificación probabilístico basado en el Teorema de Bayes:
$$P(y|x) = \frac{P(x|y)P(y)}{P(x)}$$
* $P(y|x)$: **Probabilidad a posteriori**
* $P(x|y)$: **Verosimilitud (likelihood)**
*  $P(y)$: **Probabilidad a priori**
*  $P(x)$: **Evidencia**
donde $y$ es la clase (género literario) y $x$ es el vector de características (palabras).

Su caracteristica principal es asumir ''ingenuamente'' de que los features son condicionalmente independientes entre sí, dada la clase:
$$P(x|y) = \prod_{i=1}^{n} P(x_i|y)$$
Dado el género $y$, la aparición de una palabra es independiente de la aparición de otras palabras.
Esta simplificación permite estimar las probabilidades de forma eficiente, aunque en realidad las palabras no son independientes.

##Procesamiento del catalogo

Se implementa la función de descarga que verifica si los archivos ya existen localmente antes de descargarlos desde el servidor. Se descarga el catalogo de libros en formato CSV y se carga en un DataFrame de pandas.

El catalogo contiene datos de los libros siendo el título, autor, idioma, géneros y otros campos relevantes.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import tarfile
import zipfile
import requests
import re
import pickle
import time
from pathlib import Path
from bs4 import BeautifulSoup
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from scipy.sparse import csr_matrix, issparse, save_npz, load_npz
from collections import Counter
from sklearn.preprocessing import LabelEncoder




In [2]:
URL_CATALOGO = "https://web.csc.gob.ar/~jzuloaga/epub/catalog.csv"
URL_COMPRIMIDO = "https://web.csc.gob.ar/~jzuloaga/epub/compressed.tar"
DATA_DIR = Path("./epub_data")
RUTA_CATALOGO = DATA_DIR / "catalog.csv"
RUTA_TAR = DATA_DIR / "compressed.tar"
DIR_EXTRAIDO = DATA_DIR / "books"
CACHE_DIR = DATA_DIR / "cache"

CACHE_DIR.mkdir(exist_ok=True, parents=True)
DATA_DIR.mkdir(exist_ok=True)

PADRON = 107193

# dataset reducido
USE_REDUCED = False
TAM_TRAIN = 300
TAM_TEST = 100

In [3]:
def descargar(url, ruta):
    if not ruta.exists():
        resp = requests.get(url, stream=True)
        resp.raise_for_status()
        with open(ruta, 'wb') as f:
            for chunk in resp.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f"Descarga completada: {ruta.name}")
    else:
        print(f"Archivo ya existe: {ruta.name}")

descargar(URL_CATALOGO, RUTA_CATALOGO)
descargar(URL_COMPRIMIDO, RUTA_TAR)

if not DIR_EXTRAIDO.exists() or not any(DIR_EXTRAIDO.rglob("*.epub")):
    print(f"\nExtrayendo EPUBs en {DIR_EXTRAIDO}...")
    DIR_EXTRAIDO.mkdir(exist_ok=True)
    with tarfile.open(RUTA_TAR) as tar:
        tar.extractall(path=DIR_EXTRAIDO)
    print(f"✓ Extracción completada")
else:
    print(f"✓ EPUBs ya extraídos en {DIR_EXTRAIDO}")


Descarga completada: catalog.csv
Descarga completada: compressed.tar

Extrayendo EPUBs en epub_data/books...


/tmp/ipython-input-1831754746.py:19: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=DIR_EXTRAIDO)


✓ Extracción completada


In [4]:
cat = pd.read_csv(RUTA_CATALOGO)
print("Primeras filas del catálogo:")
print(cat.head())
print("\nInformación del catálogo:")
print(cat.info())


Primeras filas del catálogo:
   EPL Id                          Título              Autor  \
0   78718            Cuando era divertido        Eloy Moreno   
1   80251       El laberinto de la bestia        Emily Rodda   
2   80612            A orillas del tiempo      Fernando Wulf   
3   80570  Memorias del mariscal de campo  Albert Kesselring   
4   77527                   El álbum ruso  Michael Ignatieff   

                          Géneros           Colección  Volumen  \
0                           Otros                 NaN      NaN   
1  Aventuras, Fantástico, Juvenil  La saga de Deltora      6.0   
2     Ciencias sociales, Historia                 NaN      NaN   
3              Historia, Memorias                 NaN      NaN   
4   Biografía, Historia, Memorias                 NaN      NaN   

   Año publicación                                           Sinopsis  \
0             2022  Quizás hay un momento en la vida para leer est...   
1             2000  Lief, Barda y Jasmine s

### Filtrado por idioma

Se filtran los libros en español del catálogo completo, lo que reduce el conjunto de datos necesaria para el análisis de datos, además evita mezclar idiomas que confundirian al vectorizador.
Luego se escanea el directorio de archivos EPUB descargados y se crea un mapeo entre los IDs del catalogo y las rutas de archivo reales. Se verifica qué libros del catálogo tienen su archivo digital disponible creando una columna booleana que indica disponibilidad. Solo se trabajara con los archivos que si están disponibles físicamente en el sistema de archivos.

In [5]:
cat_es = cat[cat['Idioma'].str.lower() == 'español'].copy()
print(f"\nTotal de libros en catálogo: {len(cat)}")
print(f"Libros en español: {len(cat_es)}")
print(f"Porcentaje en español: {len(cat_es)/len(cat)*100:.2f}%")


Total de libros en catálogo: 69193
Libros en español: 61063
Porcentaje en español: 88.25%


### Mapear con archivos disponibles

Solo el 4.68% de libros en español tienen archivo EPUB disponible, esto dice el conjunto de datos real con el cual se trabajara.


In [6]:
archivos = {int(f.stem): f for f in DIR_EXTRAIDO.rglob("*.epub") if f.stem.isdigit()}
cat_es['ruta_epub'] = cat_es['EPL Id'].map(archivos)
cat_filt = cat_es[cat_es['ruta_epub'].notna()].copy()
print(f"Con archivo digital: {len(cat_filt)}")

Con archivo digital: 8958


## Definición de las clases

Cada libro puede tener múltiples géneros, como pueden ser ''Aventuras, Fantástico, Juvenil''.
Dado un libro con géneros $$\{g_1, g_2, ..., g_k\}$$, seleccionamos:

$$g^* = \arg\max_{g \in \{g_1, ..., g_k\}} \text{count}_{\text{global}}(g)$$

Esto quiere decir que se extraen los géneros de cada libro y se realiza un análisis estadístico identificado los géneros más frecuentes.



In [7]:
def extraer_generos(texto):
    if pd.isna(texto):
        return []
    return [g.strip() for g in re.split(r'[,;]', str(texto)) if g.strip()]

cat_filt['lista_gen'] = cat_filt['Géneros'].apply(extraer_generos)
todos_gen = [g for gl in cat_filt['lista_gen'] for g in gl]

print("\nTop 20 géneros:")
for i, (gen, cant) in enumerate(Counter(todos_gen).most_common(20), 1):
    print(f"{i}. {gen}: {cant}")



Top 20 géneros:
1. Drama: 1201
2. Otros: 1059
3. Aventuras: 1000
4. Policial: 998
5. Realista: 854
6. Intriga: 689
7. Histórico: 591
8. Filosofía: 553
9. Historia: 533
10. Fantástico: 470
11. Ciencias sociales: 350
12. Humor: 348
13. Romántico: 334
14. Ciencia ficción: 304
15. Memorias: 298
16. Terror: 261
17. Psicológico: 238
18. Espiritualidad: 222
19. Juvenil: 211
20. Sátira: 198


 "Otros" es la tercer categoría con más frecuencia en el catálogo original. Sin embargo, "Otros" no constituye realmente un género literario sino una categoría residual que agrupa todo aquello que no encaja fácilmente en las géneros predefinidos. Desde la perspectiva del clasificador, esta categoría es problemática porque no tiene características distintivas propias: no existe un "vocabulario de Otros" que el modelo pueda aprender a reconocer.

 Finalmente, se aplica un filtro adicional eliminando cualquier género que tenga menos de diez ejemplos en el conjunto de datos. Este umbral mínimo tiene fundamento estadístico: con menos de diez ejemplos, no podemos estimar confiablemente las probabilidades de cinco mil palabras diferentes, y las estimaciones que obtengamos estarán dominadas por el efecto del suavizado de Laplace más que por patrones reales en los datos.

In [8]:
# Eliminar "Otros"
cat_filt['lista_clean'] = cat_filt['lista_gen'].apply(
    lambda gl: [g for g in gl if 'otro' not in g.lower()]
)
cat_filt = cat_filt[
    cat_filt['lista_clean'].str.len() > 0
].copy()

### Selección de género único por libro

Dado que muchos libros pertenecen a múltiples generos simultaneamente, se implementa un criteio determínistico para asignar un unico género a cada libro. La estrategia consiste en seleccionar el género más frecuente globalmente del dataset cuando un libro tiene múltiples categorias. Esto se justifica ya que al favorecer los géneros más frecuentes, maximizamos el tamaño de nuestras clases principales, lo que mejora la capacidad del modelo para estimar correctamente los parámetros estadísticos de esas clases. Géneros con muy pocos ejemplos conducen a estimaciones poco confiables de las probabilidades condicionales, especialmente cuando trabajamos con vocabularios de miles de palabras.

 Se calcula un ranking global de frecuencias de géneros y se aplica la función de selección, finalmente se filtran géneros con menos de diez ejemplos para evitar problemas de datos insuficientes durante el entrnamiento.




In [9]:
# Selección de género único
all_clean = [g for gl in cat_filt['lista_clean'] for g in gl]
ranking_gen = Counter(all_clean)

def elegir_genero(lista):
    return max(lista, key=lambda g: ranking_gen.get(g, 0))

cat_filt['genero'] = cat_filt['lista_clean'].apply(elegir_genero)

# Filtrar géneros con <10 ejemplos
MIN_SAMPLES = 10
conteo_gen = cat_filt['genero'].value_counts()
gen_validos = conteo_gen[conteo_gen >= MIN_SAMPLES].index
cat_final = cat_filt[
    cat_filt['genero'].isin(gen_validos)
].copy()


# Estadísticas
multi_gen = cat_filt['lista_clean'].apply(len) > 1
print(f"\n✓ Libros con múltiples géneros: {multi_gen.sum()}")
print(f"✓ Libros con un solo género: {(~multi_gen).sum()}")

conteo_final = cat_filt['genero'].value_counts()
print(f"\nTotal de géneros: {len(conteo_final)}")
print(f"Total de libros: {len(cat_filt)}")

for i, (genre, count) in enumerate(conteo_final.head(20).items(), 1):
    pct = count / len(cat_filt) * 100
    print(f"{i:2d}. {genre:30s} {count:6d} ({pct:5.2f}%)")



✓ Libros con múltiples géneros: 2350
✓ Libros con un solo género: 5813

Total de géneros: 54
Total de libros: 8163
 1. Drama                            1201 (14.71%)
 2. Policial                          978 (11.98%)
 3. Aventuras                         947 (11.60%)
 4. Realista                          685 ( 8.39%)
 5. Filosofía                         553 ( 6.77%)
 6. Historia                          506 ( 6.20%)
 7. Histórico                         371 ( 4.54%)
 8. Fantástico                        364 ( 4.46%)
 9. Humor                             277 ( 3.39%)
10. Ciencias sociales                 253 ( 3.10%)
11. Intriga                           242 ( 2.96%)
12. Ciencia ficción                   238 ( 2.92%)
13. Memorias                          222 ( 2.72%)
14. Romántico                         151 ( 1.85%)
15. Terror                            139 ( 1.70%)
16. Espiritualidad                    112 ( 1.37%)
17. Viajes                            105 ( 1.29%)
18. Infantil     

### División en conjuntos de Train y Test

Se realiza la división del dataset en conjuntos de train y test usando el número de padrón como semilla para reproducibilidad.

La estratificación garantiza que la distribución de clases se mantenga aproximadamente constante entre los conjuntos de entrenamiento y prueba. Matemáticamente, esto significa que $$\frac{N_y^{\text{train}}}{N^{\text{train}}} \approx \frac{N_y^{\text{test}}}{N^{\text{test}}} \approx \frac{N_y^{\text{total}}}{N^{\text{total}}}$$ para cada género $y$.

In [10]:
train_cat, test_cat = train_test_split(
    cat_final,
    test_size=0.25,
    random_state=PADRON,
    stratify=cat_final['genero']
)

print(f"\nTrain: {len(train_cat)} | Test: {len(test_cat)}")
print(f"\nDistribución en TRAIN (top 10):")
for genre, count in train_cat['genero'].value_counts().head(10).items():
    print(f"  {genre:30s} {count:5d}")

print(f"\nDistribución en TEST (top 10):")
for genre, count in test_cat['genero'].value_counts().head(10).items():
    print(f"  {genre:30s} {count:5d}")


Train: 6085 | Test: 2029

Distribución en TRAIN (top 10):
  Drama                            901
  Policial                         733
  Aventuras                        710
  Realista                         514
  Filosofía                        415
  Historia                         379
  Histórico                        278
  Fantástico                       273
  Humor                            208
  Ciencias sociales                190

Distribución en TEST (top 10):
  Drama                            300
  Policial                         245
  Aventuras                        237
  Realista                         171
  Filosofía                        138
  Historia                         127
  Histórico                         93
  Fantástico                        91
  Humor                             69
  Ciencias sociales                 63


## Preprocesamiento de texto

Se implementa la función que extrae el textoo plano de los archivos EPUB, que son esencialmente archivos ZIP conteniendo HTML/XHTML. La función descomprime el EPUB, busca todos los archivos con extensiones HTML, extrae el contenido usando BeautifulSoup eliminando las etiquetas HTML, y concatena todo el textoo.
Luego se define un generador que permite procesar los libros bajo demanda, lo cual es eficiente para datasets grandes.

In [11]:
def extraer_texto(ruta):
    try:
        with zipfile.ZipFile(ruta) as z:
            texto = []
            for f in z.namelist():
                if f.endswith(('.xhtml', '.html', '.htm')):
                    try:
                        soup = BeautifulSoup(z.read(f), 'html.parser')
                        texto.append(soup.get_text(separator=' ', strip=True))
                    except:
                        continue
            return ' '.join(texto)
    except:
        return ""

In [12]:
if USE_REDUCED:
    print(f"\n⚠️ MODO DESARROLLO: Usando dataset reducido")
    print(f"   Train: {TAM_TRAIN} libros | Test: {TAM_TEST} libros")
    train_cat = train_cat.head(TAM_TRAIN).copy()
    test_cat = test_cat.head(TAM_TEST).copy()
else:
    print(f"\n✓ MODO PRODUCCIÓN: Usando dataset completo")

print(f"   Train final: {len(train_cat)} | Test final: {len(test_cat)}")


✓ MODO PRODUCCIÓN: Usando dataset completo
   Train final: 6085 | Test final: 2029


In [ ]:
# ===== SISTEMA DE CACHÉ COMPLETO ===== ← CORREGIDO
RUTA_VECT = CACHE_DIR / "vectorizer.pkl"
RUTA_TRAIN = CACHE_DIR / "X_train.npz"
RUTA_TEST = CACHE_DIR / "X_test.npz"
RUTA_ETIQ = CACHE_DIR / "labels.pkl"

def procesar_datos():
    le = LabelEncoder()
    inicio_total = time.time()

    # ===== PROCESAR TRAIN  =====
    inicio = time.time()

    textos_train, etiq_train = [], []

    for i, (_, fila) in enumerate(train_cat.iterrows(), 1):

        texto = extraer_texto(fila['ruta_epub'])
        if texto.strip():
            textos_train.append(texto)
            etiq_train.append(fila['genero'])

    # Entrenar vectorizador CON los textoos ya extraídos
    vectorizer = CountVectorizer(
        max_df=0.8,
        min_df=5,
        stop_words=None,
        max_features=5000,
        lowercase=True,
        token_pattern=r'\b[a-záéíóúñ]{3,}\b'
    )

    vectorizer.fit(textos_train)

    X_train = vectorizer.transform(textos_train)
    y_train = le.fit_transform(etiq_train)

    del textos_train  # Liberar memoria

    # ===== PROCESAR TEST =====
    inicio = time.time()

    textos_test, etiq_test, test_indices = [], [], []

    for i, (idx, fila) in enumerate(test_cat.iterrows(), 1):
        texto = extraer_texto(fila['ruta_epub'])
        if texto.strip():
            textos_test.append(texto)
            etiq_test.append(fila['genero'])
            test_indices.append(idx)

    X_test = vectorizer.transform(textos_test)
    y_test = le.transform(etiq_test)

    del textos_test

    # ===== GUARDAR TODO =====
    if not USE_REDUCED:
        print("💾 Guardando en caché...")

        with open(RUTA_VECT, 'wb') as f:
            pickle.dump(vectorizer, f)

        save_npz(RUTA_TRAIN, X_train)
        save_npz(RUTA_TEST, X_test)

        with open(RUTA_ETIQ, 'wb') as f:
            pickle.dump({
                'y_train': y_train,
                'y_test': y_test,
                'test_indices': test_indices,
                'label_encoder': le
            }, f)


    return vectorizer, X_train, X_test, y_train, y_test, test_indices, le

# ===== CARGAR O PROCESAR =====
cache_exists = (RUTA_VECT.exists() and
                RUTA_TRAIN.exists() and
                RUTA_TEST.exists() and
                RUTA_ETIQ.exists())

if cache_exists and not USE_REDUCED:
    inicio = time.time()

    with open(RUTA_VECT, 'rb') as f:
        vectorizer = pickle.load(f)

    X_train = load_npz(RUTA_TRAIN)
    X_test = load_npz(RUTA_TEST)

    with open(RUTA_ETIQ, 'rb') as f:
        labels = pickle.load(f)

    y_train = labels['y_train']
    y_test = labels['y_test']
    test_indices = labels['test_indices']
    le = labels['label_encoder']

else:
    vectorizer, X_train, X_test, y_train, y_test, test_indices, le = procesar_datos()


### Vectorización con CountVectorizer

Se implementa la transformación de textoo a representación numérica. Se configura *CountVectorizer* con estos parámetros:

* max_df=0.8: Elimina palabras que aparecen en >80% de los documentos. Son palabras demasiado comunes que no ayudan a discriminar géneros.
  
* min_df=5: Requiere que una palabra aparezca en al menos 5 documentos.
  
* stop_words='spanish': Elimina palabras funcionales del español (el, la, de, que...) que no aportan información semántica sobre el género literario.
  
* max_features=5000: Limita el vocabulario para eficiencia computacional manteniendo las palabras más informativas.

 El vectorizador se entrena con el conjunto de entrenamiento usando el generador, construyendo así el vocabulario que se usará para todos los documentos.


### Análisis de obras clásicas

Se realiza un análisis comparando dos libros de géneros diferentes para visualizar cómo el vectorizador captura las diferencias léxicas. Se buscan obras clásicas conocidas en el catálogo, se extrae su textoo completo, y se vectorizan. Luego se identifican las cuarenta palabras más frecuentes que son únicas de cada libro, es decir, que aparecen en uno pero no en el otro. Este análisis revela vocabulario distintivo, nombres propios característicos, temáticas específicas y diferencias en estilo narrativo entre géneros. Estas diferencias léxicas son precisamente las características que el clasificador Naive Bayes aprenderá a explotar para distinguir entre géneros literarios.




 La salida del código muestra el análisis de "El horror de Dunwich" de H.P. Lovecraft, categorizado como Terror, versus "Orgullo y prejuicio" de Jane Austen, categorizado como Romántico. Esta comparación no podría ser más apropiada: Lovecraft representa la literatura de terror cósmico del siglo veinte, mientras que Austen ejemplifica la novela de costumbres romántica del siglo diecinueve. Los mundos literarios que habitan estos autores son tan diferentes que esperaríamos vocabularios completamente disjuntos, y el análisis confirma esta hipótesis de manera contundente.


Cuando el modelo se entrena en múltiples libros de Terror, aprende que palabras como "monstruo", "terror", "ruidos" tienen alta probabilidad condicional dado Terror: $$P(\text{monstruo}|\text{Terror}) \gg P(\text{monstruo}|\text{Romántico})$$. Similarmente, aprende que $$P(\text{matrimonio}|\text{Romántico}) \gg P(\text{matrimonio}|\text{Terror})$$. Cuando llega un nuevo feature para clasificar, el modelo calcula para cada género la probabilidad conjunta $$P(y) \prod_i P(w_i|y)^{n_i}$$, donde $n_i$ es cuántas veces aparece cada palabra $w_i$. Si el feature contiene muchas instancias de "monstruo", "terror" y "ruidos", el producto de probabilidades será mucho mayor para la clase Terror que para Romántico, llevando a la clasificación correcta.




In [ ]:
def find_book(catalog, search_term):
    matches = catalog[catalog['Título'].str.contains(search_term, case=False, na=False)]
    return matches.iloc[0] if not matches.empty else None

search_terms_1 = ['horror']
search_terms_2 = ['orgullo']

libro1 = None
for term in search_terms_1:
    libro1 = find_book(cat_final, term)
    if libro1 is not None:
        break
if libro1 is None:
    libro1 = cat_final.iloc[0]

libro2 = None
for term in search_terms_2:
    libro2 = find_book(cat_final, term)
    if libro2 is not None:
        break
if libro2 is None:
    libro2 = cat_final.iloc[-1]

print(f"\nLibro 1: {libro1['Título']}")
print(f"  Autor: {libro1['Autor']}")
print(f"  Género: {libro1['genero']}")

print(f"\nLibro 2: {libro2['Título']}")
print(f"  Autor: {libro2['Autor']}")
print(f"  Género: {libro2['genero']}")

texto1 = extraer_texto(libro1['ruta_epub'])
texto2 = extraer_texto(libro2['ruta_epub'])
vectors = vectorizer.transform([texto1, texto2]).toarray()

vocab = np.array(vectorizer.get_feature_names_out())
unique_1 = (vectors[0] > 0) & (vectors[1] == 0)
unique_2 = (vectors[1] > 0) & (vectors[0] == 0)

for libro_num, unique_mask, libro_row in [(1, unique_1, libro1), (2, unique_2, libro2)]:
    print(f"\nTOP 40 PALABRAS ÚNICAS - LIBRO {libro_num}: {libro_row['Título']}")
    print("-"*60)

    counts = vectors[libro_num-1] * unique_mask
    top_idx = counts.argsort()[-40:][::-1]

    for rank, idx in enumerate(top_idx, 1):
        if counts[idx] > 0:
            print(f"{rank:2d}. {vocab[idx]:25s} {int(vectors[libro_num-1][idx]):5d}")

## Multinomial Naive Bayes

Se implementa desde cero la clase Multinomial NB que aplica el teorema de Bayes con la suposición de independencia condicional entre features. El método *fit* calcula las probabilidades a priori de cada clase y las probabilidades condicionales de cada palabra dado cada género, aplicando suavizado de Laplace con parámetro $\alpha$ para evitar probabilidades cero. Los cálculos se realizan en espacio logarítmico para evitar underflow numérico. El método *predict_proba* calcula las probabilidades posteriores de cada clase usando la log-likelihood conjunta y normalizando con el truco log-sum-exp para estabilidad numérica. El método *predict* simplemente retorna la clase con mayor probabilidad posterior. Esta implementación sigue fielmente la formulación matemática de Naive Bayes multinomial:
$$P(y|x)∝ P(y) ∏_i P(x_i|y)$$
donde $P(x_i|y)$ se estima con suavizado de Laplace :
$$P(x_i|y) = (count(x_i, y) + alpha) / (count(y) + alpha * n_features)$$



La estimación de los parámetros del modelo se realiza mediante máxima verosimilitud sobre el conjunto de entrenamiento. La probabilidad a priori de cada clase se estima simplemente como la frecuencia relativa de esa clase en los datos de entrenamiento: $$P(y) = \frac{N_y}{N_{\text{total}}}$$, donde $N_y$ es el número de documentos de clase $y$ y $N_{\text{total}}$ es el total de documentos.

Si estimamos directamente $$P(w_i|y) = \frac{c_{i,y}}{\sum_j c_{j,y}}$$, donde $c_{i,y}$ es el número total de veces que aparece la palabra $i$ en todos los documentos de clase $y$, enfrentamos el problema de que algunas palabras no aparecerán en absoluto en ciertos géneros durante el entrenamiento. Cuando esto ocurre, $P(w_i|y) = 0$, lo que causa que la probabilidad conjunta de todo el documento sea cero debido a la multiplicación: $P(x|y) = \prod_i P(w_i|y)^{n_i} = 0$. Esto es claramente inaceptable porque una sola palabra no vista haría imposible clasificar un documento bajo ese género, independientemente de cuántas otras palabras sean consistentes con él.

La solución estándar a este problema es el **suavizado de Laplace**. Añadimos un pseudo-conteo de $\alpha$ a cada palabra en cada clase, como si hubiéramos observado cada palabra $\alpha$ veces adicionales más de lo que realmente observamos. La fórmula modificada se convierte en $$P(w_i|y) = \frac{c_{i,y} + \alpha}{\sum_j c_{j,y} + \alpha V}$$. El denominador se ajusta sumando $\alpha V$ porque hemos añadido $\alpha$ a cada una de las $V$ palabras del vocabulario. Con $\alpha = 1.0$, que es el valor estándar de Laplace, garantizamos que $P(w_i|y) > 0$ para todas las palabras y todos los géneros.

In [ ]:
class MultinomialNB:
    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def fit(self, X, y):
        if issparse(X):
            X = X.toarray()

        self.classes_ = np.unique(y)
        n_classes = len(self.classes_)
        n_features = X.shape[1]

        class_counts = np.zeros(n_classes)
        feature_counts = np.zeros((n_classes, n_features))

        for idx, c in enumerate(self.classes_):
            mask = (y == c)
            class_counts[idx] = mask.sum()
            feature_counts[idx] = X[mask].sum(axis=0)

        self.class_log_prior_ = np.log(class_counts) - np.log(class_counts.sum())
        smoothed = feature_counts + self.alpha
        totals = smoothed.sum(axis=1).reshape(-1, 1)
        self.feature_log_prob_ = np.log(smoothed) - np.log(totals)

        return self

    def _joint_log_likelihood(self, X):
        if issparse(X):
            X = X.toarray()
        return X @ self.feature_log_prob_.T + self.class_log_prior_

    def predict_proba(self, X):
        log_proba = self._joint_log_likelihood(X)
        log_max = log_proba.max(axis=1, keepdims=True)
        exp_proba = np.exp(log_proba - log_max)
        return exp_proba / exp_proba.sum(axis=1, keepdims=True)

    def predict(self, X):
        return self.classes_[self._joint_log_likelihood(X).argmax(axis=1)]

mnb = MultinomialNB(alpha=1.0)
mnb.fit(X_train, y_train)
print("✓ Modelo entrenado")

#### Preparación de datos

Se prepara el dataset codificando las etiquetas de género a valores numéricos usando LabelEncoder. Se itera sobre los catálogos de entrenamiento y prueba extrayendo el textoo completo de cada EPUB y vectorizándolo con el CountVectorizer previamente entrenado. Solo se incluyen documentos que tienen contenido textoual no vacío. El resultado son matrices dispersas *X_train* y *X_test* conteniendo los conteos de palabras, y vectores *y_train* e *y_test* con las etiquetas codificadas. Esta representación numérica es la entrada que recibirá el clasificador.

La codificación de las etiquetas de género mediante *LabelEncoder* asigna un número entero único a cada género: Drama podría ser cero, Aventuras uno, Policial dos, etc.
El mismo encoder debe aplicarse a los conjuntos de train y test para garantizar que el mismo número represente el mismo género en ambos. Además, necesitamos el encoder para la operación inversa: después de que el modelo predice un número, usamos inverse_transform para convertirlo de vuelta al nombre del género.

#### Entrenando modelo

Se instancia el clasificador Multinomial NB con $\alpha$ igual a 1 y se entrena con el conjunto de entrenamiento mediante el método *fit*. Durante el entrenamiento, el modelo calcula las estadísticas necesarias: cuenta cuántos documentos pertenecen a cada clase, suma los conteos de cada palabra por clase, y calcula las probabilidades logarítmicas a priori y condicionales.

In [ ]:
mnb = MultinomialNB(alpha=1.0)
mnb.fit(X_train, y_train)

### Evaluar modelo

Se realizan predicciones sobre los conjuntos de entrenamiento y prueba, y se calculan métricas de rendimiento. Se reportan accuracy  y macro F1-score  para ambos conjuntos. Estas métricas permiten cuantificar objetivamente qué tan bien el modelo ha aprendido a distinguir entre géneros literarios.

In [ ]:
y_train_pred = mnb.predict(X_train)
y_test_pred = mnb.predict(X_test)

train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)
train_f1 = f1_score(y_train, y_train_pred, average='macro')
test_f1 = f1_score(y_test, y_test_pred, average='macro')

print(f"\n{'Métrica':<20} {'Entrenamiento':>15} {'Testeo':>15}")
print("-" * 52)
print(f"{'Accuracy':<20} {train_acc:>15.4f} {test_acc:>15.4f}")
print(f"{'Macro F1':<20} {train_f1:>15.4f} {test_f1:>15.4f}")

dummy_error = 1 - (test_cat['genero'].value_counts().max() / len(test_cat))
print(f"\nError clasificador dummy: {dummy_error:.4f}")

Un clasificador dummy que siempre predice la clase mayoritaria obtendría un accuracy de veintiuno por ciento. Nuestro modelo con cuarenta y dos por ciento supera esto, demostrando que sí ha aprendido algo útil más allá de simplemente memorizar la clase más común. Sin embargo, solo veintiún puntos porcentuales sobre el baseline. Un modelo bien generalizado en un problema de cuarenta y cuatro clases debería lograr accuracy sustancialmente mayor.

### Errores de clasificación

Se identifican todos los casos donde el modelo clasificó incorrectamente en el conjunto de prueba. Para cada error se recupera el libro correspondiente del catálogo junto con el género verdadero y el género predicho. Se calculan las confianzas de predicción para entender si el modelo estaba seguro o dudoso en sus errores. Se ordenan los errores por popularidad del libro y se muestran los diez errores más notables.


In [ ]:
errors_mask = y_test != y_test_pred
error_idx = np.where(errors_mask)[0]

print(f"\nTotal de errores: {len(error_idx)} de {len(y_test)} ({errors_mask.mean()*100:.2f}%)")

if len(error_idx) > 0:
    error_data = []

    for i in error_idx:
        original_idx = test_indices[i]
        row = test_cat.loc[original_idx]
        proba = mnb.predict_proba(X_test[i:i+1])[0]

        error_data.append({
            'Título': row['Título'],
            'Autor': row['Autor'],
            'Género_Real': le.inverse_transform([y_test[i]])[0],
            'Género_Predicho': le.inverse_transform([y_test_pred[i]])[0],
            'Confianza': proba.max(),
            'Descargas': row.get('Descargas', 0)
        })

    df_errors = pd.DataFrame(error_data)

    if df_errors['Descargas'].sum() > 0:
        df_errors = df_errors.sort_values('Descargas', ascending=False)
        metric = "descargas"
    else:
        df_errors = df_errors.sort_values('Confianza', ascending=False)
        metric = "confianza de predicción"

    print(f"\nTop 10 errores (por {metric}):\n")
    for i, row in enumerate(df_errors.head(10).itertuples(), 1):
        print(f"{i}. {row.Título}")
        print(f"   Autor: {row.Autor}")
        print(f"   Género real: {row.Género_Real}")
        print(f"   Predicho: {row.Género_Predicho}")
        print(f"   Confianza: {row.Confianza:.1%}")
        if row.Descargas > 0:
            print(f"   Descargas: {row.Descargas:,}")
        print()


    print(f"Confianza promedio: {df_errors['Confianza'].mean():.1%}")
    print(f"Confianza mínima: {df_errors['Confianza'].min():.1%}")
    print(f"Confianza máxima: {df_errors['Confianza'].max():.1%}")
    print(f"\nErrores con confianza >80%: {(df_errors['Confianza'] > 0.8).sum()}")
    print(f"Errores con confianza >90%: {(df_errors['Confianza'] > 0.9).sum()}")
    print(f"Errores con confianza >95%: {(df_errors['Confianza'] > 0.95).sum()}")


Las estadísticas de confianza revelan que prácticamente todos los errores  tienen confianza superior al noventa y cinco por ciento. Esto indica que no estamos viendo casos ambiguos donde el modelo duda entre dos géneros similares y elige incorrectamente; estamos viendo casos donde el modelo está sistemáticamente sesgado hacia ciertos géneros basado en características superficiales del vocabulario.

### Causas Sistemáticas de Error

El análisis cualitativo de errores revela varias causas sistemáticas que merecen discusión teórica.  Muchos libros pertenecen  a múltiples géneros simultáneamente. Cuando forzamos estos libros en una única categoría mediante nuestro criterio de selección basado en frecuencia global, creamos etiquetas que son arbitrarias en cierto sentido. El modelo que aprende de estas etiquetas está intentando encontrar patrones en datos que contienen ruido.

La confusión entre géneros semánticamente cercanos es otra fuente importante de errores. Géneros como Drama y Realista, o Aventuras y Policial, comparten vocabulario .  Géneros con alta similitud de vocabulario son intrínsecamente difíciles de distinguir para un modelo basado únicamente en frecuencias de palabras.

 Al representar un documento como simplemente un vector de conteos de palabras, perdemos toda información sobre la estructura narrativa, el desarrollo de la trama, el arco de personajes, el tono emocional, el registro lingüístico y la intención autoral.

El desbalance de clases introduce sesgos sistemáticos que afectan tanto el entrenamiento como la evaluación.



### Mejoras Teóricas y Direcciones Futuras

El uso de TF-IDF  en lugar de conteos brutos modificaría cómo pesamos las palabras. La fórmula $$\text{TF-IDF}(w,d) = \frac{n_{w,d}}{\sum_{w'} n_{w',d}} \cdot \log\frac{N}{|\{d : w \in d\}|}$$ penaliza palabras que aparecen en muchos documentos (como palabras comunes) y realza palabras que aparecen en pocos documentos (palabras raras y específicas). Esto automáticamente da más peso a palabras discriminativas sin necesidad de ajustar manualmente max_df y min_df.

Agrupando categorías semánticamente similares abordaría simultáneamente varios problemas. Podríamos crear super-categorías como Ficción (agrupando Drama, Realista, Romántico), Aventura (Aventuras, Policial, Intriga), Fantástico (Fantástico, Ciencia ficción, Terror), y No-ficción (Historia, Filosofía, Memorias). Esto reduciría el número de clases de cuarenta y cuatro a quizás ocho o diez, aumentando el número de ejemplos por clase.
